<font color=#009afa size=50>DAT5001 - Visualisation of Data</font>

<font size=50>1 - Cleaning Satellite Dataset for Presentation</font>

|**Assessment Code:** |PRES1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5001_S2_25|
|**Course Tutor:**|Dr. Imtiaz Khan|


## Initialise Notebook

In [6]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [7]:
# Standard Libraries
import json
import random
from datetime import datetime
import sys

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.pyplot import figure
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# GeoPy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Pycountry 
import pycountry

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Beautifulsoup
from bs4 import BeautifulSoup

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Check for Internet Access

In [8]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

In [9]:
# Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.

# Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online


_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.planet4589.org',
                  'https://datascience.daemonchild.com']

if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    sys.exit('Stopping notebook running here.')


[INTERNET] Internet Connectivity Detected. Using Internet resources.


## Load the Datasets from the Internet

Reference Website:
https://www.planet4589.org/space/gcat/index.html

Tab Separated Value Files: 

- SatCat: https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv
- Events: https://planet4589.org/space/gcat/tsv/cat/ecat.tsv
- Payload: https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv   
- Org Data: https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv
- Launch sites: https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv

Documentation: https://www.planet4589.org/space/gcat/pdf/doc.pdf

Reference: "McDowell, J, 2020: General Catalog of Artificial Space Objects, https://planet4589.org/space/gcat", First accessed 24/10/2025

> VIII.2.1.1 The Standard (S) Satellite Catalog
>
> Since the late 1950s the US government has maintained a catalog of satellites (often referred to as the SATCAT). Each
satellite is given a sequential number (1 was the rocket stage from the first Sputnik). These numbers are sometimes
referred to as SSN numbers.

In [10]:
# This will get the very latest dataset from John McDowells's page. This dataset is updated daily!
satcat_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv', sep='\t')
events_df = pd.read_csv('https://planet4589.org/space/gcat/tsv/cat/ecat.tsv', sep='\t')
pay_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv', sep='\t')
orgs_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv', sep='\t')
lsite_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv', sep='\t')

datasets = {'satcat_df': satcat_df, 
            'events_df': events_df, 
            'pay_df': pay_df, 
            'orgs_df': orgs_df,
            'lsite_df': lsite_df
            }

### Explore these datasets

In [11]:
for k,v in datasets.items():
    print ("-----")
    print ("Dataset: " + k)
    print ("")
    print (v.columns)
    print ("")
    print (v.shape)

-----
Dataset: satcat_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object')

(66650, 42)
-----
Dataset: events_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object

## Data Cleaning
Remove columns that are not useful for the report.

**Satcat Database**

In [12]:
# Remove columns we can't use
columns_to_keep = ['Piece', 'Type', 'Name', 'LDate', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass', 'Length', 'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee']
sdf = satcat_df[columns_to_keep]

# Rename Columns for easier reading
sdf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date', 'Owner':'Operator Code','Manufacturer':'Manufacturer Code'}, inplace=True)

**Events Database**

In [13]:
edf_columns_to_keep = ['Piece', 'LDate', 'DDate', 'Status', 'Perigee', 'Apogee']
edf = events_df[edf_columns_to_keep]

edf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date'}, inplace=True)

**Organisations Database**

In [14]:
# Create the CombinedName column using owner_EName if available, otherwise owner_Name
orgs_df['CombinedName'] = orgs_df['EName'].fillna(orgs_df['Name'])

# Remove columns we can't use
orgs_columns_to_keep = ['#Code', 'ShortName', 'CombinedName', 'Longitude', 'Latitude', 'Parent']
odf = orgs_df[orgs_columns_to_keep]

# Rename Columns for easier reading
odf.rename(columns={'#Code':'Org Code', 'ShortName':'Short Name', 'CombinedName':'Long Name'}, inplace=True)

Convert Latitude and Longitude to City and State values

In [15]:

import time

# Set up the geocoder
geolocator = Nominatim(user_agent="geoapiExercises")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# Define a function to extract city and state
def get_location_info(lat, lon):
    time.sleep(1.1)
    print (".", end="", flush=True)
    try:
        location = reverse((lat, lon), language='en')
        if location and location.raw.get('address'):
            address = location.raw['address']
            city = address.get('city', address.get('town', address.get('village', '')))
            state = address.get('state', '')

            return f"{city}|{state}"
    except:
        return None
    
#filtered_odf = odf.tail(10)    

# Apply the function using a lambda
#filtered_odf['CityState'] = filtered_odf.apply(lambda row: get_location_info(row['Latitude'], row['Longitude']), axis=1)

**Payload Database**

In [16]:
payload_columns_to_keep = ['Piece', 'Program', 'Class', 'Category']
pdf = pay_df[payload_columns_to_keep]

# Rename 'Piece' to 'CosparID'
pdf.rename(columns={'Piece':'CosparID'}, inplace=True)

## Combine the Datasets

In [17]:
# Rename odf columns for Operator merge
owner_odf = odf.rename(columns=lambda x: f'Operator {x}' if x != 'Org Code' else 'Operator Code')
sdf = sdf.merge(owner_odf, on='Operator Code', how='left')

# Repeat for Manufacturer merge
manufacturer_odf = odf.rename(columns=lambda x: f'Manufacturer {x}' if x != 'Org Code' else 'Manufacturer Code')
sdf = sdf.merge(manufacturer_odf, on='Manufacturer Code', how='left')

# Merge payload data
sdf = sdf.merge(pdf, on='CosparID', how='left')

The events database contains updates to the Satcat data, so these need to be applied like a database log to get the updated values.

For example, this ID shows as being docked.

In [18]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19117    DK
Name: Status, dtype: object

However, later on in the events database, it undocked (UDK) and then finally deorbited (D):

In [19]:
edf[edf['CosparID'] == '1988-038A']['Status']

767    UDK
770      D
Name: Status, dtype: object

In [20]:
sdf['Launch Date'] = sdf['Launch Date'].replace("-", np.nan)  # Shouldn't be any!
sdf = sdf.dropna(subset=['Launch Date'])
#sdf['Decay Date'] = sdf['Decay Date'].replace("-", np.nan)

# There are some dates that are questionable - remove the question marks
sdf['Decay Date'] = sdf['Decay Date'].str.replace('?', '', regex=False)

# Ensure types
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], errors='coerce')

edf = edf.copy()
edf['Launch Date'] = pd.to_datetime(edf['Launch Date'], errors='coerce')
edf['Decay Date'] = pd.to_datetime(edf['Decay Date'], errors='coerce')

# Dedupe edf to the latest update per CosparID by index, therefore keeping the last update.
edf_latest = (
    edf.sort_index()                # ensure index order is respected
       .drop_duplicates('CosparID', keep='last')
)

# Merge and update only specified columns
merged = sdf.merge(edf_latest, on='CosparID', how='left', suffixes=('', '_update'))

# Columns to update from edf_latest
cols_to_update = ['Launch Date', 'Decay Date', 'Status', 'Perigee', 'Apogee']

for col in cols_to_update:
    update_col = f"{col}_update"
    if update_col in merged.columns:
        merged[col] = merged[update_col].combine_first(merged[col])

# Clean up _update columns
merged = merged.drop(columns=[c for c in merged.columns if c.endswith('_update')])

# Final updated dataframe
sdf = merged



Check with the example from above:

In [21]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    D
Name: Status, dtype: object

# Feature Engineering

In [22]:
# These mappings were created from the documentation:

class_json = {
  "A": "Non-commercial",        # Academic / Amateur / Non-Profit
  "B": "Commercial",
  "C": "Government (Civil)",
  "D": "Government (Defence)",  # and Military and Intelligence
  "U": "Unknown"
}

type_json = {
  "P": "Payload",
  "C": "Component",
  "R": "Rocket Body",
  "D": "Debris",
  "S": "Suborbital payload",
  "X": "Deleted Object",
  "Z": "Spurious Entry"
}

category_json = {
  "AST": "Astronomy",
  "BIO": "Biology",
  "CAL": "Calibration",
  "COM": "Communications",
  "EDU": "Education",
  "EOSCI": "Earth observing science",
  "EW": "Missile early warning",
  "GEOD": "Geodesy",
  "IMG": "Imaging (optical)",
  "IMG-R": "Imaging (radar)",
  "INF": "Infrastructure",
  "MET": "Meteorology (imaging)",
  "MET-RO": "Meteorology (radio occultation)",
  "MGRAV": "Microgravity experiments",
  "MISC": "Miscellaneous",
  "NAV": "Navigation",
  "PLAN": "Deep space related mission",
  "RB": "Rocket body",
  "RV": "Reentry vehicle",
  "SCI": "Scientific studies",
  "SIG": "Signals intelligence",
  "SS": "Spaceship",
  "TARG": "Target",
  "TECH": "Technology",
  "WEAPON": "Weapon",
  "UNK": "Unknown"
}

status_json = {
    "O": "In orbit",
    "AO": "Attached in orbit",
    "AO IN": "Attached inside",
    "UDK": "Undocked",
    "REL": "Released",
    "DEP": "Deployed",
    "TO": "Takeoff Launch",
    "TOA": "Attached at takeoff",
    "OX": "In orbit",
    "E": "Exploded",
    "N": "Renamed",
    "NA": "Renamed attached",
    "LEASE": "Lease",
    "S": "Targeted suborbital entry/impact",
    "R": "Reentered",
    "D": "Deorbited",
    "L": "Landed",
    "LF": "Failed landing",
    "AS": "Attached Suborbital reentered",
    "AR": "Reentered attached",
    "AR IN": "Reentered inside",
    "AL": "Landed attached",
    "AL IN": "Landed inside",
    "TX": "Transmission ended",
    "F": "Failed",
    "AF": "Failed attached",
    "DK": "Docked",
    "GRP": "Grappled",
    "ATT": "Attached",
    "TFR": "Transfer Inside",
    "TFR E": "Transfer external Inside",
    "TFR IN": "Transfer in"
}

### Category, Class and Type Fields

In [23]:
# Split the Category column at the first '/'
split_cols = sdf['Category'].str.split('/', n=1, expand=True)

# Remove trailing spurious characters from each part
sdf['Category'] = sdf['Category'].fillna('UNK')
sdf['Primary Category'] = split_cols[0].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[0] > 1 else 'UNK'
sdf['Secondary Category'] = split_cols[1].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[1] > 1 else 'UNK'

# Type field - Remove Sub-Types
sdf['Type'] = sdf['Type'].str[0]

# Remove X and Z objects ("Spurious and unknown")
sdf = sdf[~sdf['Type'].isin(['X', 'Z'])]

# Class if not labelled
sdf['Class'] = sdf['Class'].fillna('U')

# Map the descriptions for better data visualisation labelling
sdf['Class Desc'] = sdf['Class'].map(class_json)
sdf['Type Desc'] = sdf['Type'].map(type_json)
sdf['Primary Category Desc'] = sdf['Primary Category'].map(category_json)
sdf['Secondary Category Desc'] = sdf['Secondary Category'].map(category_json)
sdf['Status Desc'] = sdf['Status'].map(status_json)

In [24]:
sdf.drop(columns=['Type', 'Class', 'Category', 'Primary Category', 'Secondary Category', 'Status'], inplace=True, axis=1)
sdf.rename(columns={'Type Desc':'Type'}, inplace=True)

# Rename columns ending with " Desc" to just 
sdf = sdf.rename(
    columns=lambda c: c[:-5] if c.endswith(" Desc") else c
)

In [25]:
sdf['Type'].value_counts()

Type
Debris         27825
Payload        24635
Component       7905
Rocket Body     6195
Name: count, dtype: int64

In [26]:
sdf['Class'].value_counts()

Class
Unknown                 41837
Commercial              15246
Government (Defence)     5420
Government (Civil)       3154
Non-commercial            883
Name: count, dtype: int64

In [27]:
sdf['Primary Category'].value_counts()

Primary Category
Communications                     15097
Imaging (optical)                   2998
Technology                          1570
Calibration                          811
Spaceship                            765
Signals intelligence                 701
Navigation                           623
Scientific studies                   521
Meteorology (imaging)                303
Imaging (radar)                      268
Deep space related mission           257
Astronomy                            218
Missile early warning                213
Earth observing science              117
Geodesy                               65
Meteorology (radio occultation)       63
Weapon                                37
Microgravity experiments              31
Biology                               30
Target                                21
Reentry vehicle                       10
Education                              4
Name: count, dtype: int64

In [28]:
sdf['Status'].value_counts()

Status
Reentered             32218
In orbit              31634
Landed                 1326
Deorbited               401
Reentered attached      300
Attached in orbit        86
Landed attached          24
Lease                    14
Exploded                 12
Landed inside             3
Transfer Inside           3
Released                  2
Reentered inside          1
Renamed                   1
Name: count, dtype: int64

Work out whether the object is still in orbit by combining specific status wording and whether there is a Decayed Date recorded or not.

In [29]:
# Define category lists
not_in_orbit_categories = ['Reentered', 'Landed', 'Deorbited', 'Reentered attached', 'Reentered inside', 'Landed attached', 'Landed inside']
in_orbit_categories = ['In Orbit', 'Attached in orbit', 'Docked']

# Create the new column 'In Orbit'
sdf['In Orbit'] = (
    (sdf['Status'].isin(in_orbit_categories) & sdf['Decay Date'].isna()) |     # In Orbit and no decay date
    (~sdf['Status'].isin(not_in_orbit_categories) & sdf['Decay Date'].isna())  # Not in the "not in orbit" list and no decay date
)

Returning to the example from before:

In [30]:
sdf[sdf['CosparID'] == '1988-038A']['In Orbit']

19116    False
Name: In Orbit, dtype: bool

In [31]:
sdf.sample(20)[['CosparID','Status', 'Decay Date','In Orbit']]

,CosparID,Status,Decay Date,In Orbit
4185,1969-095A,Reentered attached,1969-11-08 22:40:00,False
20556,1990-029J,In orbit,NaT,True
58109,2023-166K,In orbit,NaT,True
65527,2025-203W,In orbit,NaT,True
64047,2025-110H,In orbit,NaT,True
43664,2018-083D,Reentered,NaT,False
53061,1982-092CBG,Reentered,NaT,False
51839,2022-022V,In orbit,NaT,True
20390,1989-101A,In orbit,NaT,True
65908,2025-228V,In orbit,NaT,True


In [32]:
sdf['In Orbit'].value_counts().reset_index()

,In Orbit,count
0,False,34744
1,True,31816


In [33]:
# Check the results looks sane
filtered_df = sdf[(sdf['Type'] == 'Payload')]
                  
pivot_table = filtered_df.pivot_table(
    index='Status',
    columns='In Orbit',
    aggfunc='size',
    fill_value=0
)

# Display the result
print(pivot_table)

In Orbit            False  True 
Status                          
Attached in orbit      14     71
Deorbited             362      0
Exploded                8      0
In orbit               23  16699
Landed               1325      0
Landed attached        24      0
Landed inside           3      0
Lease                   6      8
Reentered            5449      0
Reentered attached    298      0
Released                1      0
Renamed                 1      0
Transfer Inside         0      1


### Generate Additional Date Values

The dates that are available are:

- Launch Date (always available)
- Decay Date (may be NULL or '?')

These values are converted from strings in the base dataset into python datetime objects.
This allows for comparisons and for additional values ot be inferred from them.

The following date columns were created:

- Launch Year
- Launch Month (Numerical)
- Launch Month (Text)
- Launch Decade (1950's: is 1950-1959.)
- Decay Year
- Decay Month (Numerical)
- Decay Month (Text)
- Decay Decade 

A temporary value for 'Effective Decay Date' is either copied from 'Decay Date' or set to 'Today' if the object is "In Orbit".
This allows for lifetime values in days, months and years to be calculated.

Finally, the datetime objects are reverted to strings (in UK format).

In [34]:
# Modify date columns
# Convert to datetime format
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], format='%Y %b %d', errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], format='%Y %b %d', errors='coerce')

# Create a Year of Launch and Decay
sdf['Launch Year'] = sdf['Launch Date'].dt.year.astype('Int64')
sdf['Decay Year'] = sdf['Decay Date'].dt.year.astype('Int64')
sdf['Launch Month No'] = sdf['Launch Date'].dt.month.astype('Int64')
sdf['Decay Month No'] = sdf['Decay Date'].dt.month.astype('Int64')
sdf['Launch Month'] = sdf['Launch Date'].dt.strftime('%B')
sdf['Decay Month'] = sdf['Decay Date'].dt.strftime('%B')
sdf['Launch Decade'] = ((sdf['Launch Year'] // 10) * 10)
sdf['Decay Decade'] = ((sdf['Decay Year'] // 10) * 10)
# Replace null Decay Date with today's date
today = pd.Timestamp(datetime.today())
#sdf['Effective Decay Date'] = sdf['Decay Date'].fillna(today)
# If Status == "In Orbit", set to today, else use Decay Date
sdf['Effective Decay Date'] = np.where(
    sdf['Status'] == "In Orbit",
    today,
    sdf['Decay Date']
)

# Convert to datetime 
sdf['Effective Decay Date'] = pd.to_datetime(sdf['Effective Decay Date'], errors='coerce')

# Calculate lifetime
delta = sdf['Effective Decay Date'] - sdf['Launch Date']
sdf['Lifespan Days'] = delta.dt.days
sdf['Lifespan Months'] = (delta.dt.days / 30.44).round(1)  # Approximate month length
sdf['Lifespan Years'] = (delta.dt.days / 365.25).round(2)  # Account for leap years

# Format as dd/mm/yyyy (because we are British)
sdf['Launch Date'] = sdf['Launch Date'].dt.strftime('%d/%m/%Y')
sdf['Decay Date'] = sdf['Decay Date'].dt.strftime('%d/%m/%Y')

# Drop Effective Decay Date column
sdf = sdf.drop('Effective Decay Date', axis=1)

### Map Country Code to Country Name

The country codes ('State') supplied are two figure codes. 
These are first converted to three figure codes, then a Country name derived from this.

In [35]:
# Convert alpha-2 to alpha-3
def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

# Apply conversion to your State column
sdf['Country Code'] = sdf['State'].apply(alpha2_to_alpha3)
sdf['Country'] = sdf['Country Code'].apply(lambda x: pycountry.countries.get(alpha_3=x).name if x else None)

# Drop State Column
sdf.drop(columns='State', axis=1, inplace=True)

In [36]:
# Create a combined "Program Root" field
#sdf["Program Root"] = sdf["Program"].str.split().str[0]
sdf["Program Root"] = sdf["Program"].str.extract(r"^([A-Za-z0-9]+)")

In [37]:
# Reorder columns alphabetically
sdf = sdf.reindex(sorted(sdf.columns), axis=1)

In [38]:
sdf.columns

Index(['Apogee', 'Class', 'CosparID', 'Country', 'Country Code', 'Decay Date',
       'Decay Decade', 'Decay Month', 'Decay Month No', 'Decay Year',
       'Diameter', 'In Orbit', 'Launch Date', 'Launch Decade', 'Launch Month',
       'Launch Month No', 'Launch Year', 'Length', 'Lifespan Days',
       'Lifespan Months', 'Lifespan Years', 'Manufacturer Code',
       'Manufacturer Latitude', 'Manufacturer Long Name',
       'Manufacturer Longitude', 'Manufacturer Parent',
       'Manufacturer Short Name', 'Mass', 'Name', 'Operator Code',
       'Operator Latitude', 'Operator Long Name', 'Operator Longitude',
       'Operator Parent', 'Operator Short Name', 'Perigee', 'Primary Category',
       'Program', 'Program Root', 'Secondary Category', 'Shape', 'Span',
       'Status', 'Type'],
      dtype='object')

### Write it out for further use



In [39]:
timestamp_formatted = today.strftime("%Y%m%d")

# Create filenames, one timestamped, one 'latest'
filepath = '../../datasets/'
filename = f'{filepath}space_objects_{timestamp_formatted}.csv'
latestname = f'{filepath}space_objects_latest.csv'

# Write out CSV files
sdf.to_csv(filename, index=False)
sdf.to_csv(latestname, index=False)